# One hundred parallel BM4 trajectories: base-cell escape study

This notebook tests whether any of **100 independent guiding-centre trajectories** crosses the square boundary of the potential's fundamental periodic cell. Each trajectory is integrated with `BM4Implicit1` for 100 normalized cycles using 20 complete BM4 steps per cycle.

The potential is periodic, so crossing the displayed square is not a physical loss of the trajectory: it means that the continuous, unwrapped coordinate has moved to another periodic image of the base cell. The escape diagnostic deliberately uses those unwrapped coordinates.

## Execution note

The integrations run in separate spawned processes. Every worker is restricted to one internal numerical thread to avoid CPU oversubscription. Run the notebook from the beginning; the long cell reports progress every 30 seconds and after each completed trajectory.

In [ ]:
import os
from pathlib import Path
from types import SimpleNamespace

from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np

from diagnostics.paths import find_project_root
from potential import load_gc2d_h5_potential
from studies import latin_hypercube_gc_configuration
from studies.bm4_parallel_recurrence import (
    ParallelBM4RecurrenceConfig,
    run_parallel_bm4_recurrence,
)
from visualization import display_records_table

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## Reproducible campaign configuration

In [ ]:
# Measured, nondimensionalized GC2D potential.
project_root = find_project_root(Path.cwd())
data_path = project_root / "data/potential/V1/PHI_2.h5"
magnetic_field = 1.5
characteristic_length = 0.06
mode_selection = (0, 1)
interpolation_order = 3

# Reproducible spatial coverage of the base cell.
particle_count = 100
initial_condition_seed = 20260906
domain_margin_fraction = 0.05

# Requested BM4 campaign. Saving every step is essential: an orbit may
# leave the square and return between two integer cycle boundaries.
cycle_count = 100
steps_per_cycle = 20
saved_samples_per_cycle = steps_per_cycle
t_span = (0.0, float(cycle_count))
rho = 0.3
coupling_frequency = float(np.pi / 8.0)
newton_absolute_tolerance = 1e-12
newton_relative_tolerance = 1e-11
newton_max_iterations = 40
jacobian_relative_step = float(np.cbrt(np.finfo(float).eps))

# Keep this below the number of physical CPU cores if memory is limited.
available_cpu_count = os.cpu_count() or 1
worker_count = min(8, available_cpu_count, particle_count)

if not data_path.is_file():
    raise FileNotFoundError(f"Measured HDF5 potential not found: {data_path}")
potential = load_gc2d_h5_potential(
    data_path,
    B=magnetic_field,
    characteristic_length=characteristic_length,
    indx=mode_selection,
    interpolation_order=interpolation_order,
)
initial_configuration = latin_hypercube_gc_configuration(
    potential,
    particle_count=particle_count,
    seed=initial_condition_seed,
    domain_margin_fraction=domain_margin_fraction,
)
config = ParallelBM4RecurrenceConfig(
    particle_count=particle_count,
    t_span=t_span,
    steps_per_cycle=steps_per_cycle,
    saved_samples_per_cycle=saved_samples_per_cycle,
    rho=rho,
    coupling_frequency=coupling_frequency,
    absolute_tolerance=newton_absolute_tolerance,
    relative_tolerance=newton_relative_tolerance,
    max_iterations=newton_max_iterations,
    jacobian_relative_step=jacobian_relative_step,
    worker_count=worker_count,
    progress=True,
)
assert config.cycle_count == 100
assert config.steps_per_cycle == 20
assert config.step_count == 2_000
assert config.output_sample_count == 2_001
assert config.integration_step == 0.05
display(Markdown(
    f"**Resolved campaign:** `{particle_count}` trajectories, "
    f"`{config.step_count}` steps per trajectory, "
    f"$\\Delta t={config.integration_step:g}$, "
    f"`{config.output_sample_count}` saved states per trajectory and "
    f"`{config.worker_count}` worker processes."
))

## Initial-condition coverage and escape boundary

The black rectangle is the half-open fundamental cell $[x_{min}, x_{min}+L) \times [y_{min}, y_{min}+L)$. Initial points are sampled with a 5% margin from that boundary.

In [ ]:
initial_state = initial_configuration.initial_state
assert initial_state is not None
initial_x, initial_y = initial_configuration.positions(initial_state)
grid = potential.grid
period = float(grid.period)
x_bounds = (float(grid.xmin), float(grid.xmin + period))
y_bounds = (float(grid.ymin), float(grid.ymin + period))

field_at_zero = np.asarray(potential.evaluate(0.0), dtype=float)
figure, axis = plt.subplots(figsize=(8, 7), constrained_layout=True)
image = axis.imshow(
    field_at_zero.T,
    origin="lower",
    extent=(*x_bounds, *y_bounds),
    cmap="RdBu_r",
    aspect="equal",
)
axis.scatter(
    initial_x, initial_y, c=np.arange(1, particle_count + 1),
    cmap="viridis", s=28, edgecolor="black", linewidth=0.3,
)
axis.add_patch(Rectangle(
    (x_bounds[0], y_bounds[0]), period, period, fill=False,
    color="black", linewidth=1.5,
))
axis.set(
    title="Initial conditions for the BM4 escape campaign",
    xlabel="$x$", ylabel="$y$", xlim=x_bounds, ylim=y_bounds,
)
figure.colorbar(image, ax=axis, label="$\\Phi(0,x,y)$")
plt.show()

## Parallel BM4 integrations

This is the long-running cell. It launches one independent task per trajectory and retains all 2001 step-boundary positions needed by the escape audit.

In [ ]:
result = run_parallel_bm4_recurrence(
    potential,
    initial_configuration,
    config=config,
)

## Result and numerical-work audit

In [ ]:
assert result.positions.shape == (particle_count, 2, config.output_sample_count)
assert result.initial_positions.shape == (particle_count, 2)
assert result.runtime_seconds.shape == (particle_count,)
np.testing.assert_allclose(
    result.times,
    np.arange(config.output_sample_count, dtype=float) * config.integration_step,
)
assert np.all(result.maximum_residual_to_tolerance <= 1.0)
display(Markdown(
    f"All **{particle_count} trajectories** completed in "
    f"**{result.wall_runtime_seconds:.1f} s wall time** with "
    f"{config.worker_count} worker processes."
))

## Detect the first crossing of the square

A saved state is outside when either unwrapped coordinate is below its lower boundary or greater than or equal to its upper boundary. Because every complete BM4 step is saved, the reported first-escape time is resolved to $\Delta t=0.05$.

In [ ]:
x = result.positions[:, 0, :]
y = result.positions[:, 1, :]
outside = (
    (x < x_bounds[0]) | (x >= x_bounds[1])
    | (y < y_bounds[0]) | (y >= y_bounds[1])
)
escaped = np.any(outside, axis=1)
escaped_particles = np.flatnonzero(escaped)
first_escape_indices = np.full(particle_count, -1, dtype=int)
first_escape_indices[escaped] = np.argmax(outside[escaped], axis=1)

# Integer periodic-image indices quantify how far an unwrapped orbit travels.
cell_x = np.floor((x - x_bounds[0]) / period).astype(int)
cell_y = np.floor((y - y_bounds[0]) / period).astype(int)
maximum_cell_distance = np.max(
    np.maximum(np.abs(cell_x), np.abs(cell_y)), axis=1,
)

def crossed_sides(particle, sample):
    """Return every base-cell side crossed at one saved state."""
    sides = []
    if x[particle, sample] < x_bounds[0]:
        sides.append("left")
    if x[particle, sample] >= x_bounds[1]:
        sides.append("right")
    if y[particle, sample] < y_bounds[0]:
        sides.append("bottom")
    if y[particle, sample] >= y_bounds[1]:
        sides.append("top")
    return " + ".join(sides)

escape_rows = tuple(
    SimpleNamespace(
        trajectory=int(particle + 1),
        initial_x=result.initial_positions[particle, 0],
        initial_y=result.initial_positions[particle, 1],
        first_escape_step=int(first_escape_indices[particle]),
        first_escape_time=result.times[first_escape_indices[particle]],
        first_escape_x=x[particle, first_escape_indices[particle]],
        first_escape_y=y[particle, first_escape_indices[particle]],
        side=crossed_sides(particle, first_escape_indices[particle]),
        maximum_cell_distance=int(maximum_cell_distance[particle]),
    )
    for particle in escaped_particles
)

if escape_rows:
    display_records_table(
        escape_rows,
        columns=(
            ("trajectory", "Trajectory", "d"),
            ("initial_x", "Initial x", ".6f"),
            ("initial_y", "Initial y", ".6f"),
            ("first_escape_step", "First escape step", "d"),
            ("first_escape_time", "First escape time", ".3f"),
            ("first_escape_x", "Escape x", ".6f"),
            ("first_escape_y", "Escape y", ".6f"),
            ("side", "Crossed side", "s"),
            ("maximum_cell_distance", "Max image index", "d"),
        ),
    )
else:
    display(Markdown("**No trajectory left the base-cell square.**"))

## Unwrapped trajectories and cumulative escapes

The left panel shows every unwrapped path. Escaping paths are red and their first detected outside state is marked with a black cross. The right panel shows how many distinct trajectories have escaped by each saved time.

In [ ]:
figure, (path_axis, count_axis) = plt.subplots(
    1, 2, figsize=(15, 6.5), constrained_layout=True,
)
for particle in range(particle_count):
    color = "tab:red" if escaped[particle] else "tab:blue"
    alpha = 0.58 if escaped[particle] else 0.20
    path_axis.plot(x[particle], y[particle], color=color, alpha=alpha, linewidth=0.7)
path_axis.scatter(
    result.initial_positions[:, 0], result.initial_positions[:, 1],
    color="tab:green", s=10, alpha=0.7, label="Initial positions", zorder=4,
)
if escaped_particles.size:
    first_indices = first_escape_indices[escaped_particles]
    path_axis.scatter(
        x[escaped_particles, first_indices], y[escaped_particles, first_indices],
        color="black", marker="x", s=30, linewidth=1.0,
        label="First outside sample", zorder=5,
    )
path_axis.add_patch(Rectangle(
    (x_bounds[0], y_bounds[0]), period, period, fill=False,
    color="black", linewidth=1.6, label="Base-cell square",
))
path_axis.set(
    title="Unwrapped BM4 trajectories", xlabel="$x$", ylabel="$y$",
)
path_axis.set_aspect("equal", adjustable="datalim")
path_axis.legend(loc="best")

first_escape_times = np.full(particle_count, np.inf)
first_escape_times[escaped] = result.times[first_escape_indices[escaped]]
cumulative_escaped = np.sum(
    first_escape_times[:, None] <= result.times[None, :], axis=0,
)
count_axis.step(
    result.times, cumulative_escaped, where="post", color="tab:red",
    linewidth=1.8,
)
count_axis.set(
    title="Cumulative first escapes", xlabel="Time / normalized cycles",
    ylabel="Distinct escaped trajectories", xlim=config.t_span,
    ylim=(-0.5, particle_count + 0.5),
)
plt.show()

## Coordinate envelopes

These panels make isolated boundary crossings visible even when the planar plot is crowded. The shaded band is the base-cell interval; solid curves show the minimum and maximum unwrapped coordinate over all 100 trajectories.

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
for axis, values, bounds, coordinate in (
    (axes[0], x, x_bounds, "x"),
    (axes[1], y, y_bounds, "y"),
):
    axis.axhspan(*bounds, color="tab:green", alpha=0.10, label="Base-cell interval")
    axis.plot(result.times, np.min(values, axis=0), color="tab:blue", label=f"min {coordinate}")
    axis.plot(result.times, np.max(values, axis=0), color="tab:orange", label=f"max {coordinate}")
    axis.axhline(bounds[0], color="black", linewidth=0.9)
    axis.axhline(bounds[1], color="black", linewidth=0.9)
    axis.set_ylabel(f"${coordinate}$")
    axis.legend(loc="best")
axes[0].set_title("Coordinate envelope of all unwrapped trajectories")
axes[1].set_xlabel("Time / normalized cycles")
plt.show()

## Data-driven campaign summary

In [ ]:
escape_count = int(np.sum(escaped))
if escape_count:
    earliest_particle = int(escaped_particles[np.argmin(first_escape_times[escaped])])
    earliest_time = float(first_escape_times[earliest_particle])
    escape_statement = (
        f"trajectory {earliest_particle + 1} first crossed the square at "
        f"time `{earliest_time:.3f}`"
    )
else:
    escape_statement = "no trajectory crossed the square"
display(Markdown("\n".join((
    f"- **Escaped trajectories:** {escape_count}/{particle_count}.",
    f"- **Earliest event:** {escape_statement}.",
    f"- **Largest periodic-image index reached:** {int(np.max(maximum_cell_distance))}.",
    f"- **Parallel wall time:** {result.wall_runtime_seconds:.1f} s with {config.worker_count} workers.",
    f"- **Sum of isolated worker runtimes:** {np.sum(result.runtime_seconds):.1f} s.",
))))